# 🥈 04_ingestao_silver — Camada Silver de `ecommerce_pedidos`
**Squad 2 · Dupla 3** · Passo 6 do plano · regras 2, 3, 4, 5 e 10 da planilha

Lê da Bronze os micro-lotes ainda não processados, padroniza e aplica, pedido a pedido, **só as regras da planilha**.
Os pedidos vão para dois destinos, ambos em Delta, modo `append`, com a coluna `silver_processed_at`:

| Destino | Conteúdo |
|---|---|
| `squad2/silver/ecommerce_pedidos` | pedidos aprovados, padronizados |
| `squad2/quarantine/ecommerce_pedidos` | pedidos reprovados, padronizados, com `motivo_quarentena` |

| Regra da planilha | Como a Silver aplica |
|---|---|
| 2 · não processar o mesmo arquivo duas vezes | marca d'água + retirada do que já foi gravado (pedido + micro-lote) |
| 3 · `valor_total` > 0 e decimal | `DECIMAL(12,2)`; vazio ou ≤ 0 → quarentena |
| 4 · `dt_pedido` é data e hora | vazio → quarentena (o tipo já foi garantido na Bronze) |
| 5 · `status_pedido` nos valores permitidos | vazio ou fora da lista → quarentena (`ERROR` é dado defeituoso, decisão com o Geovany) |
| 10 · `dt_ultima_atualizacao_status` < `dt_pedido` | → quarentena: o dado corrompido não chega à Silver nem à Gold (o alerta sai na Gold) |

O que a planilha não pede **não reprova** ninguém: campo vazio fora dessas regras, método de pagamento desconhecido e `id_pedido` repetido
em micro-lotes diferentes são só monitorados (⚠️). Nenhum valor é corrigido ou inventado; o original continua na Bronze.

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração e contrato da EDA |
| 2 | Leitura do pendente na Bronze (marca d'água) |
| 3 | Padronização |
| 4 | Regras de qualidade (2, 3, 4, 5 e 10) |
| 5 | Gravação em `append` e avanço da marca d'água |
| 6 | Conferência e monitoramento |
| 7 | Resumo para o orquestrador |

Cada etapa **faz** o trabalho e **valida** em seguida: ✅ ok · ⚠️ achado (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` em modo rápido e, do contrato da EDA, o fuso das datas e a lista de status permitidos (regra 5).
**Valida:** se a Bronze existe e se o contrato foi carregado.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from functools import reduce
from pyspark.sql import functions as F

contrato = ler_metadata("eda_raw") or {}
FUSO_HORAS = int(contrato.get("fuso", {}).get("horas_ate_utc", 0))     # horas a somar para converter as datas em UTC
STATUS_PERMITIDOS = contrato.get("enums", {}).get("status_pedido", [])  # regra 5 (ERROR fica de fora: decisão com o Geovany)
PAGAMENTOS_CONHECIDOS = contrato.get("enums", {}).get("metodo_pagamento", [])   # só para o monitoramento da Etapa 6: não reprova
ANOTACAO_CONTROLE = "silver_controle"                                   # metadata/ecommerce_pedidos/silver_controle.json
PARTICAO = ["data_pedido_local"]                                        # dia do pedido no horário de Brasília

# ------------------------------ validação ------------------------------
checar(tabela_delta_existe(CAMINHO_BRONZE), "Bronze encontrada", "Bronze não existe: rode o 03_ingestao_bronze antes")
checar(bool(STATUS_PERMITIDOS), f"Regra 5: {len(STATUS_PERMITIDOS)} status permitidos ({', '.join(STATUS_PERMITIDOS)})",
       "Contrato da EDA sem a lista de status: rode a EDA (Etapa 9.2) antes")
checar("fuso" in contrato, f"Fuso das datas: somar {FUSO_HORAS} h para UTC", "Fuso não encontrado no contrato: datas mantidas como estão", parar=False)

✅ Bronze encontrada
✅ Regra 5: 5 status permitidos (PROCESSANDO, PAGAMENTO APROVADO, CANCELADO, PENDENTE, AGUARDANDO)
✅ Fuso das datas: somar 3 h para UTC


## Etapa 2 — Leitura do pendente na Bronze
**Faz:** a marca d'água em `metadata/` guarda o `bronze_ingested_at` mais recente já processado. Pendente = linhas da Bronze ingeridas depois dela.
Cada rodada da Bronze é um commit único com um único `bronze_ingested_at`, então a marca separa os micro-lotes sem ambiguidade e tem tamanho fixo.
**Valida:** se há pendente; sem pendente, a rodada termina com sucesso e zero linhas.

In [0]:
controle = ler_metadata(ANOTACAO_CONTROLE) or {}
marca_dagua = controle.get("marca_dagua")                              # None na primeira execução: processa a Bronze inteira

pendente = ler_delta(CAMINHO_BRONZE).drop("ano", "mes", "dia", "hora")  # partição da Bronze não interessa à Silver
if marca_dagua:
    pendente = pendente.filter(F.col("bronze_ingested_at") > F.to_timestamp(F.lit(marca_dagua)))
lote = pendente.agg(F.count("*").alias("linhas"), F.max("bronze_ingested_at").alias("nova_marca")).first()
arquivos_lote = [r["bronze_source_file"] for r in pendente.select("bronze_source_file").distinct().collect()]
resumo = {"linhas_lidas": lote["linhas"], "arquivos": len(arquivos_lote), "aprovadas": 0, "quarentena": 0,
          "ja_gravadas": 0, "por_motivo": {}, "marca_dagua": marca_dagua}

# ------------------------------ validação ------------------------------
print(f"Marca d'água atual: {marca_dagua or 'nenhuma (primeira execução)'}")
checar(True, f"{lote['linhas']} linhas pendentes de {len(arquivos_lote)} micro-lotes", "")

Marca d'água atual: 2026-10-07 13:57:05.806411
✅ 0 linhas pendentes de 0 micro-lotes


## Etapa 3 — Padronização
**Faz:** limpeza e tratamento do Passo 6, sem reprovar ninguém (a padronização vem antes das regras, para o motivo apontar só o problema real):

| Coluna | Transformação |
|---|---|
| `status_pedido` | maiúsculas, sem espaços nas pontas (assim `" cancelado"` é comparado como `CANCELADO` na regra 5) |
| `metodo_pagamento` | minúsculas, sem espaços nas pontas |
| `valor_total`, `valor_frete` | `DECIMAL(12,2)`: o tipo decimal da regra 3 (dinheiro não usa número aproximado); valor que não cabe vira vazio e cai na regra 3, em vez de derrubar a rodada |
| `dt_pedido`, `dt_previsao_entrega`, `dt_ultima_atualizacao_status` | convertidas para UTC com o fuso do contrato |
| `data_pedido_local` (nova) | dia do pedido no horário de Brasília; partição da Silver |
| `bronze_source_file`, `bronze_ingested_at` | mantidas para rastrear a origem de cada linha |

**Valida:** nenhuma linha ganha ou perdida.

In [0]:
para_utc = lambda c: (F.col(c) + F.expr(f"INTERVAL {FUSO_HORAS} HOURS")).alias(c)

# Num único select, todas as expressões leem a coluna original: data_pedido_local usa dt_pedido ainda em horário local
padronizado = pendente.select(
    "id_pedido", "id_cliente", "id_endereco_entrega",
    para_utc("dt_pedido"),
    F.upper(F.trim("status_pedido")).alias("status_pedido"),
    # try_cast: em modo ANSI, um valor fora de DECIMAL(12,2) derrubaria a rodada inteira; assim ele vira vazio e cai na regra 3
    F.expr("try_cast(valor_total AS DECIMAL(12,2))").alias("valor_total"),
    F.expr("try_cast(valor_frete AS DECIMAL(12,2))").alias("valor_frete"),
    F.lower(F.trim("metodo_pagamento")).alias("metodo_pagamento"),
    para_utc("dt_previsao_entrega"), para_utc("dt_ultima_atualizacao_status"),
    F.to_date("dt_pedido").alias("data_pedido_local"),
    "bronze_source_file", "bronze_ingested_at")

# ------------------------------ validação ------------------------------
checar(padronizado.count() == lote["linhas"], f"{lote['linhas']} linhas padronizadas (nenhuma ganha ou perdida)", "Linhas alteradas na padronização")

✅ 0 linhas padronizadas (nenhuma ganha ou perdida)


## Etapa 4 — Regras de qualidade (regras 2, 3, 4, 5 e 10)
**Faz:** primeiro a **regra 2**: as linhas deste lote que já estão em algum destino (mesmo `id_pedido` e mesmo micro-lote, de uma rodada interrompida)
saem do lote, para nada ser gravado duas vezes. Depois marca, pedido a pedido, cada regra violada; o nome da regra vira o motivo, e um pedido pode ter vários.

| Motivo | Condição de reprovação |
|---|---|
| `regra3_valor_total_nao_positivo` | `valor_total` vazio ou ≤ 0 |
| `regra4_dt_pedido_invalida` | `dt_pedido` vazio (sem data e hora válida) |
| `regra5_status_fora_do_enum` | `status_pedido` vazio ou fora da lista permitida |
| `regra10_atualizacao_antes_do_pedido` | `dt_ultima_atualizacao_status < dt_pedido` |

No Spark, comparar com vazio dá "desconhecido" e não reprova; por isso as regras 3, 4 e 5 tratam o vazio de forma explícita.
**Valida:** já gravadas + aprovadas + quarentena = lidas.

In [0]:
# Regra 2: linhas deste lote já gravadas numa rodada interrompida não são processadas de novo (chave: pedido + micro-lote)
CHAVE_LINHA = ["id_pedido", "bronze_source_file"]
gravadas = [ler_delta(c).filter(F.col("bronze_source_file").isin(arquivos_lote)).select(*CHAVE_LINHA)
            for c in (CAMINHO_SILVER, CAMINHO_QUARANTINE) if arquivos_lote and tabela_delta_existe(c)]
a_processar = padronizado
if gravadas:
    ja = reduce(lambda a, b: a.unionByName(b), gravadas)
    a_processar = padronizado.join(ja, [padronizado[c].eqNullSafe(ja[c]) for c in CHAVE_LINHA], "left_anti")
n_a_processar = a_processar.count()
resumo["ja_gravadas"] = lote["linhas"] - n_a_processar

# Só as regras da planilha: nome do motivo -> condição de reprovação
REGRAS = {
    "regra3_valor_total_nao_positivo": F.col("valor_total").isNull() | (F.col("valor_total") <= 0),
    "regra4_dt_pedido_invalida": F.col("dt_pedido").isNull(),
    "regra5_status_fora_do_enum": F.col("status_pedido").isNull() | ~F.col("status_pedido").isin(STATUS_PERMITIDOS),
    "regra10_atualizacao_antes_do_pedido": F.col("dt_ultima_atualizacao_status") < F.col("dt_pedido"),
}
motivo = F.concat_ws(", ", *[F.when(cond, F.lit(nome)) for nome, cond in REGRAS.items()])
classificado = (a_processar.withColumn("motivo_quarentena", F.when(motivo != "", motivo))
                .withColumn("silver_processed_at", F.current_timestamp()))

contagem = classificado.agg(F.sum(F.col("motivo_quarentena").isNull().cast("int")).alias("aprovadas"),
                            *[F.sum(F.when(c, 1).otherwise(0)).alias(n) for n, c in
                              {"_quarentena": F.col("motivo_quarentena").isNotNull(),
                               **{n: F.col("motivo_quarentena").contains(n) for n in REGRAS}}.items()]).first().asDict()
resumo.update(aprovadas=contagem["aprovadas"] or 0, quarentena=contagem["_quarentena"] or 0,
              por_motivo={n: contagem[n] or 0 for n in REGRAS if contagem[n]})

# ------------------------------ validação ------------------------------
checar(resumo["ja_gravadas"] + resumo["aprovadas"] + resumo["quarentena"] == lote["linhas"],
       f"{resumo['ja_gravadas']} já gravadas + {resumo['aprovadas']} aprovadas + {resumo['quarentena']} na quarentena = {lote['linhas']} lidas",
       "A classificação perdeu linhas")
for nome, qtd in resumo["por_motivo"].items():
    checar(False, "", f"{nome}: {qtd} pedidos → quarentena", parar=False)

✅ 0 já gravadas + 0 aprovadas + 0 na quarentena = 0 lidas


## Etapa 5 — Gravação em `append`
**Faz:** grava aprovados na Silver e reprovados na quarentena (Delta, `append`, partição `data_pedido_local`) e só depois avança a marca d'água.
Se o notebook falhar antes de a marca avançar, a próxima rodada relê o mesmo lote, e a Etapa 4 já retira o que foi gravado (regra 2):
o `append` fica seguro contra reprocessamento.
**Valida:** releitura da marca d'água.

In [0]:
aprovados = classificado.filter(F.col("motivo_quarentena").isNull()).drop("motivo_quarentena")
reprovados = classificado.filter(F.col("motivo_quarentena").isNotNull())
n_aprovados, n_reprovados = resumo["aprovadas"], resumo["quarentena"]

if n_aprovados:
    escrever_delta(aprovados, CAMINHO_SILVER, "append", particionar_por=PARTICAO)
if n_reprovados:
    escrever_delta(reprovados, CAMINHO_QUARANTINE, "append", particionar_por=PARTICAO)
if lote["linhas"]:                                          # a marca só avança depois que os dois destinos foram gravados
    nova_marca = str(lote["nova_marca"])
    escrever_metadata(ANOTACAO_CONTROLE, {"marca_dagua": nova_marca, "atualizado_em_utc": datetime.now(timezone.utc).isoformat(),
                                          "ultima_rodada": {k: v for k, v in resumo.items() if k != "marca_dagua"}})
    resumo["marca_dagua"] = nova_marca

# ------------------------------ validação ------------------------------
print(f"Gravadas agora: {n_aprovados} na Silver · {n_reprovados} na quarentena · {resumo['ja_gravadas']} já estavam gravadas")
if lote["linhas"]:
    checar((ler_metadata(ANOTACAO_CONTROLE) or {}).get("marca_dagua") == resumo["marca_dagua"],
           f"Marca d'água avançou para {resumo['marca_dagua']}", "A marca d'água não foi gravada")

Gravadas agora: 0 na Silver · 0 na quarentena · 0 já estavam gravadas


## Etapa 6 — Conferência e monitoramento
**Faz:** relê Silver e quarentena.
**Valida:** as linhas do lote estão todas num dos dois destinos, exatamente uma vez. Em seguida, **monitora** o que a planilha não pede para reprovar
(⚠️, sem parar): aprovados com `id_pedido`, `id_cliente` ou `metodo_pagamento` vazio, método de pagamento fora dos conhecidos e `id_pedido` repetido na Silver.

In [0]:
if not lote["linhas"]:
    print("Nada pendente nesta rodada: conferência dispensada.")
else:
    silver = ler_delta(CAMINHO_SILVER) if tabela_delta_existe(CAMINHO_SILVER) else None
    quarentena = ler_delta(CAMINHO_QUARANTINE) if tabela_delta_existe(CAMINHO_QUARANTINE) else None
    do_lote = lambda df: None if df is None else df.filter(F.col("bronze_source_file").isin(arquivos_lote))
    no_destino = sum(df.count() for df in (do_lote(silver), do_lote(quarentena)) if df is not None)

    # ------------------------------ validação ------------------------------
    checar(no_destino == lote["linhas"], f"As {lote['linhas']} linhas do lote estão na Silver ou na quarentena, uma vez cada",
           f"{no_destino} linhas nos destinos para {lote['linhas']} lidas")

    # Monitoramento: fora da planilha, então só aviso
    if silver is not None:
        monitor = do_lote(silver).agg(
            *[F.sum(F.col(c).isNull().cast("int")).alias(f"{c} vazio") for c in ("id_pedido", "id_cliente", "metodo_pagamento")],
            F.sum((~F.col("metodo_pagamento").isin(PAGAMENTOS_CONHECIDOS)).cast("int")).alias("pagamento fora dos conhecidos")).first().asDict()
        monitor["id_pedido repetido na Silver"] = silver.groupBy("id_pedido").count().filter("count > 1").count()
        for nome, qtd in monitor.items():
            checar(not qtd, f"Monitor · {nome}: nenhum", f"Monitor · {nome}: {qtd} (fora da planilha: aprovado mesmo assim)", parar=False)

    # ------------------------------ resultado ------------------------------
    motivos = pd.DataFrame(sorted(resumo["por_motivo"].items(), key=lambda m: -m[1]), columns=["motivo", "pedidos"])
    display(motivos) if len(motivos) else print("Nenhum pedido novo na quarentena nesta rodada.")   # display() não aceita tabela vazia
    if quarentena is not None:
        display(do_lote(quarentena).limit(10))
    print(f"Silver total: {silver.count() if silver is not None else 0:,} · quarentena total: {quarentena.count() if quarentena is not None else 0:,}")

Nada pendente nesta rodada: conferência dispensada.


## Etapa 7 — Resumo para o orquestrador
**Faz:** libera a memória e devolve o resumo da rodada em JSON (`dbutils.notebook.exit`).

In [0]:
for _v in ["pendente", "padronizado", "a_processar", "ja", "gravadas", "classificado", "aprovados", "reprovados", "silver", "quarentena"]:
    globals().pop(_v, None)
globals().pop("_v", None)

print(json.dumps(resumo, ensure_ascii=False, default=str))
dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False, default=str))